<a href="https://colab.research.google.com/github/MZiaAfzal71/Path_Weighted_Atom_Vector/blob/main/Data%20Files/Models/XGBoostModel_plus_pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ============================================================
### XGBoost pipeline for MACCS, Morgan, PWAV, and PWAV_SHAP
### ============================================================

In [1]:
!git clone https://github.com/MZiaAfzal71/Path_Weighted_Atom_Vector
%cd Path_Weighted_Atom_Vector/Data\ Files

Cloning into 'Path_Weighted_Atom_Vector'...
remote: Enumerating objects: 803, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 803 (delta 133), reused 8 (delta 8), pack-reused 585 (from 1)
Receiving objects: 100% (803/803), 30.86 MiB | 22.77 MiB/s, done.
Resolving deltas: 100% (287/287), done.
/content/Path_Weighted_Atom_Vector/Data Files


In [2]:
!pip install osfclient rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 10.2 MB/s eta 0:00:00


In [3]:
import os
import json
import numpy as np
import pandas as pd
import shap

from osfclient.api import OSF
from subprocess import run

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

In [4]:
# Replace with your OSF project ID
project_id = "p5ga2"   # e.g. from https://osf.io/abcd3/
osf = OSF()
project = osf.project(project_id)
store = project.storage("osfstorage")

desc_folder = []
for fold in store.folders:
    if fold.path.strip("/") == "Descriptors Data":
        desc_folder = fold
        break

if desc_folder is not None:
    for f in desc_folder.files:
        local_path = f.path.strip("/")
        local_dir = os.path.dirname(local_path)

        if local_dir and not os.path.exists(local_dir):
            os.makedirs(local_dir, exist_ok=True)

        with open(local_path, "wb") as out:
            f.write_to(out)

        if local_path.endswith(".zip"):
            command = f"unzip -o '{local_path}' -d '{local_dir}'"
            run(command, shell=True)
            print(f"Unzipped {local_path} -> {local_dir}")
        else:
            print(f"Downloaded {f.path} -> {local_path}")
else:
    print("Descriptors Data folder not found on OSF.")

100%|██████████| 23.8M/23.8M [00:02<00:00, 9.18Mbytes/s]


Unzipped Descriptors Data/Descriptors Data.zip -> Descriptors Data


100%|██████████| 13.5M/13.5M [00:01<00:00, 6.79Mbytes/s]


Unzipped Descriptors Data/RAW_Descriptors_PWAV.zip -> Descriptors Data


In [7]:
# ============================================================
# Config
# ============================================================

INPUT_DIR = "Descriptors Data"
OUTPUT_DIR = "XGBoost Results Revision"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROPERTY_SHEETS = ["Log VP", "MP", "BP", "LogBCF", "LogS", "LogP"]

SPLIT_MODE = "scaffold"   # "benchmark" or "scaffold"

TEST_SIZE = 0.2
RANDOM_STATE = 42
N_PCA = 64
TOP_K = 64

DESC_VARIANTS = [
    "MACCS",
    "Morgan",
    "PWAV",
    "PWAV_SHAP64",
]

GENERATE_SHAP64_INDICES = True
SAVE_ALL_PREDICTIONS = True


# ============================================================
# Helpers
# ============================================================

def safe_name(x: str) -> str:
    return str(x).replace(" ", "").replace("/", "_").replace("\\", "_")


def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None


def get_split_indices(df, split_mode="benchmark", test_size=0.2, random_state=42):
    if split_mode == "benchmark":
        train_idx = df.index[
            df["Training/Test"].astype(str).str.strip().str.lower() == "training"
        ].tolist()

        test_idx = df.index[
            df["Training/Test"].astype(str).str.strip().str.lower() == "test"
        ].tolist()

        split_info = df[["SMILES"]].copy()
        split_info["split"] = df["Training/Test"].values
        split_info["split_mode"] = "benchmark"
        return train_idx, test_idx, split_info

    if split_mode == "scaffold":
        scaffold_df = df[["SMILES"]].copy()
        scaffold_df["scaffold"] = scaffold_df["SMILES"].apply(murcko_scaffold)

        valid_mask = scaffold_df["scaffold"].notna()
        valid_idx = scaffold_df.index[valid_mask]

        if len(valid_idx) == 0:
            raise ValueError("No valid scaffolds generated.")

        groups = scaffold_df.loc[valid_idx, "scaffold"]

        gss = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=random_state,
        )

        train_pos, test_pos = next(gss.split(valid_idx, groups=groups))

        train_idx = valid_idx[train_pos].tolist()
        test_idx = valid_idx[test_pos].tolist()

        split_info = scaffold_df.copy()
        split_info["split"] = "unused"
        split_info.loc[train_idx, "split"] = "Training"
        split_info.loc[test_idx, "split"] = "Test"
        split_info["split_mode"] = "scaffold"

        return train_idx, test_idx, split_info

    raise ValueError(f"Unknown split_mode: {split_mode}")


def get_input_file(prop, desc_variant):
    safe_prop = safe_name(prop)

    if desc_variant in {"PWAV", "PWAV_SHAP64"}:
        return os.path.join(INPUT_DIR, f"{safe_prop}_PWAV_raw.parquet")

    return os.path.join(INPUT_DIR, f"{safe_prop}_{desc_variant}.parquet")


def get_feature_columns(df, prop, desc_variant):
    target_col = f"{prop}-Measured"
    all_cols = list(df.columns)

    meta_cols = {
        "Preferred_name",
        "SMILES",
        "Training/Test",
        "original_index",
        "CAS RN",
        "NAME",
        "IUPAC Name",
        f"{prop}-EPISuite Prediction",
        f"{prop}-Prediction from our model",
        target_col,
    }

    if desc_variant in {"PWAV", "PWAV_SHAP64"}:
        pwav_cols = [c for c in all_cols if c.startswith(f"{prop}_PWAV_")]
        extra_cols = [c for c in all_cols if c.startswith(f"{prop}_EXTRA_")]

        if len(pwav_cols) == 0:
            raise ValueError(f"No PWAV columns found for {prop}.")

        return {
            "target_col": target_col,
            "pwav_cols": pwav_cols,
            "extra_cols": extra_cols,
            "direct_feature_cols": None,
        }

    direct_feature_cols = [c for c in all_cols if c not in meta_cols]

    if len(direct_feature_cols) == 0:
        raise ValueError(f"No descriptor columns found for {prop} | {desc_variant}.")

    return {
        "target_col": target_col,
        "pwav_cols": None,
        "extra_cols": None,
        "direct_feature_cols": direct_feature_cols,
    }


def shap64_file_path(prop, split_mode):
    safe_prop = safe_name(prop)
    return os.path.join(
        OUTPUT_DIR,
        f"{safe_prop}_PWAV_SHAP64_{split_mode}_top{TOP_K}_indices.json",
    )


def save_shap64_indices(prop, split_mode, top_idx, mean_shap, feature_names):
    path = shap64_file_path(prop, split_mode)

    ranked = []
    order = np.argsort(mean_shap)[::-1]

    for rank, idx in enumerate(order, start=1):
        ranked.append({
            "rank": int(rank),
            "index": int(idx),
            "feature": str(feature_names[idx]),
            "mean_abs_shap": float(mean_shap[idx]),
        })

    payload = {
        "property": prop,
        "split_mode": split_mode,
        "top_k": TOP_K,
        "selected_indices": [int(i) for i in top_idx],
        "ranked_features": ranked,
    }

    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

    return path


def load_shap64_indices(prop, split_mode):
    path = shap64_file_path(prop, split_mode)

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"SHAP64 index file not found: {path}. "
            f"Run PWAV first with GENERATE_SHAP64_INDICES=True."
        )

    with open(path, "r") as f:
        payload = json.load(f)

    return np.asarray(payload["selected_indices"], dtype=int)


def build_pwav_final_features(df, prop, train_idx, test_idx, n_pca=64):
    col_info = get_feature_columns(df, prop, "PWAV")

    target_col = col_info["target_col"]
    pwav_cols = col_info["pwav_cols"]
    extra_cols = col_info["extra_cols"]

    train_df = df.loc[train_idx].copy()
    test_df = df.loc[test_idx].copy()

    y_train = train_df[target_col].to_numpy()
    y_test = test_df[target_col].to_numpy()

    X_train_core_raw = train_df[pwav_cols].to_numpy(dtype=np.float32)
    X_test_core_raw = test_df[pwav_cols].to_numpy(dtype=np.float32)

    pca_model = None

    if n_pca is not None and n_pca < X_train_core_raw.shape[1]:
        pca_model = PCA(n_components=n_pca, random_state=RANDOM_STATE)
        X_train_core = pca_model.fit_transform(X_train_core_raw)
        X_test_core = pca_model.transform(X_test_core_raw)
        pca_feature_names = [f"PC{i+1}" for i in range(X_train_core.shape[1])]
    else:
        X_train_core = X_train_core_raw
        X_test_core = X_test_core_raw
        pca_feature_names = [f"PWAV_RAW_{i}" for i in range(X_train_core.shape[1])]

    if extra_cols:
        X_train_extra = train_df[extra_cols].to_numpy(dtype=np.float32)
        X_test_extra = test_df[extra_cols].to_numpy(dtype=np.float32)

        X_train_full = np.hstack([X_train_core, X_train_extra])
        X_test_full = np.hstack([X_test_core, X_test_extra])

        feature_names = pca_feature_names + extra_cols
    else:
        X_train_full = X_train_core
        X_test_full = X_test_core
        feature_names = pca_feature_names

    return {
        "X_train_full": X_train_full,
        "X_test_full": X_test_full,
        "y_train": y_train,
        "y_test": y_test,
        "train_df": train_df,
        "test_df": test_df,
        "pca_model": pca_model,
        "feature_names": feature_names,
        "col_info": col_info,
    }


def prepare_xy(df, prop, desc_variant, train_idx, test_idx, n_pca=64):
    if desc_variant in {"PWAV", "PWAV_SHAP64"}:
        prep = build_pwav_final_features(df, prop, train_idx, test_idx, n_pca=n_pca)

        X_train = prep["X_train_full"]
        X_test = prep["X_test_full"]

        if desc_variant == "PWAV_SHAP64":
            top_idx = load_shap64_indices(prop, SPLIT_MODE)
            X_train = X_train[:, top_idx]
            X_test = X_test[:, top_idx]
            feature_names = [prep["feature_names"][i] for i in top_idx]
        else:
            feature_names = prep["feature_names"]

        prep["X_train"] = X_train
        prep["X_test"] = X_test
        prep["feature_names_used"] = feature_names

        return prep

    col_info = get_feature_columns(df, prop, desc_variant)
    target_col = col_info["target_col"]

    train_df = df.loc[train_idx].copy()
    test_df = df.loc[test_idx].copy()

    direct_cols = col_info["direct_feature_cols"]

    return {
        "X_train": train_df[direct_cols].to_numpy(dtype=np.float32),
        "X_test": test_df[direct_cols].to_numpy(dtype=np.float32),
        "y_train": train_df[target_col].to_numpy(),
        "y_test": test_df[target_col].to_numpy(),
        "train_df": train_df,
        "test_df": test_df,
        "pca_model": None,
        "feature_names_used": direct_cols,
        "col_info": col_info,
    }


def prepare_all_features(df, prop, desc_variant, prep):
    if desc_variant in {"PWAV", "PWAV_SHAP64"}:
        col_info = prep["col_info"]
        pca_model = prep["pca_model"]

        pwav_cols = col_info["pwav_cols"]
        extra_cols = col_info["extra_cols"]

        X_all_core_raw = df[pwav_cols].to_numpy(dtype=np.float32)

        if pca_model is not None:
            X_all_core = pca_model.transform(X_all_core_raw)
        else:
            X_all_core = X_all_core_raw

        if extra_cols:
            X_all_extra = df[extra_cols].to_numpy(dtype=np.float32)
            X_all = np.hstack([X_all_core, X_all_extra])
        else:
            X_all = X_all_core

        if desc_variant == "PWAV_SHAP64":
            top_idx = load_shap64_indices(prop, SPLIT_MODE)
            X_all = X_all[:, top_idx]

        return X_all

    return df[prep["col_info"]["direct_feature_cols"]].to_numpy(dtype=np.float32)


def generate_shap64_from_pwav(prop, X_train, y_train, feature_names):
    model = XGBRegressor(random_state=RANDOM_STATE)
    model.fit(X_train, y_train)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X_train)

    mean_shap = np.abs(shap_values.values).mean(axis=0)
    top_idx = np.argsort(mean_shap)[::-1][:TOP_K]

    path = save_shap64_indices(
        prop=prop,
        split_mode=SPLIT_MODE,
        top_idx=top_idx,
        mean_shap=mean_shap,
        feature_names=feature_names,
    )

    return path


def train_xgb_and_score(prop, desc_variant):
    safe_prop = safe_name(prop)
    input_file = get_input_file(prop, desc_variant)

    print(f"\nProcessing: {prop} | {desc_variant} | {SPLIT_MODE}")
    print(f"Input: {input_file}")

    if not os.path.exists(input_file):
        print(f"Skipping missing file: {input_file}")
        return None

    df = pd.read_parquet(input_file)

    train_idx, test_idx, split_info = get_split_indices(
        df,
        split_mode=SPLIT_MODE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    # Generate SHAP64 indices from PWAV before PWAV_SHAP64 run
    if desc_variant == "PWAV_SHAP64" and GENERATE_SHAP64_INDICES:
        shap_path = shap64_file_path(prop, SPLIT_MODE)

        if not os.path.exists(shap_path):
            print("Generating SHAP64 indices from full PWAV feature vector...")

            pwav_prep = prepare_xy(
                df=df,
                prop=prop,
                desc_variant="PWAV",
                train_idx=train_idx,
                test_idx=test_idx,
                n_pca=N_PCA,
            )

            generated_path = generate_shap64_from_pwav(
                prop=prop,
                X_train=pwav_prep["X_train"],
                y_train=pwav_prep["y_train"],
                feature_names=pwav_prep["feature_names_used"],
            )

            print(f"Saved SHAP64 indices: {generated_path}")

    prep = prepare_xy(
        df=df,
        prop=prop,
        desc_variant=desc_variant,
        train_idx=train_idx,
        test_idx=test_idx,
        n_pca=N_PCA,
    )

    X_train = prep["X_train"]
    X_test = prep["X_test"]
    y_train = prep["y_train"]
    y_test = prep["y_test"]

    model = XGBRegressor(random_state=RANDOM_STATE)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")
    print(f"Descriptor dim: {X_train.shape[1]}")

    # Save split info
    split_file = os.path.join(
        OUTPUT_DIR,
        f"{safe_prop}_{desc_variant}_{SPLIT_MODE}_split.csv",
    )
    split_info.to_csv(split_file, index=False)

    # Save PCA metadata
    if prep["pca_model"] is not None:
        pca_meta = {
            "property": prop,
            "descriptor": desc_variant,
            "split_mode": SPLIT_MODE,
            "n_components": int(prep["pca_model"].n_components_),
            "explained_variance_ratio_sum": float(
                np.sum(prep["pca_model"].explained_variance_ratio_)
            ),
        }

        pca_file = os.path.join(
            OUTPUT_DIR,
            f"{safe_prop}_{desc_variant}_{SPLIT_MODE}_pca.json",
        )

        with open(pca_file, "w") as f:
            json.dump(pca_meta, f, indent=2)

    # Save selected feature list
    feature_file = os.path.join(
        OUTPUT_DIR,
        f"{safe_prop}_{desc_variant}_{SPLIT_MODE}_features.json",
    )
    with open(feature_file, "w") as f:
        json.dump(
            {
                "property": prop,
                "descriptor": desc_variant,
                "split_mode": SPLIT_MODE,
                "n_features": int(X_train.shape[1]),
                "features": [str(x) for x in prep["feature_names_used"]],
            },
            f,
            indent=2,
        )

    # Save test predictions
    test_df = prep["test_df"]

    keep_cols = [c for c in ["Preferred_name", "NAME", "SMILES"] if c in test_df.columns]
    test_results = test_df[keep_cols].copy()
    test_results[f"{prop}-Measured"] = y_test
    test_results[f"{prop}_{desc_variant}_Prediction"] = y_pred
    test_results["split_mode"] = SPLIT_MODE

    test_pred_file = os.path.join(
        OUTPUT_DIR,
        f"{safe_prop}_{desc_variant}_{SPLIT_MODE}_test_predictions.xlsx",
    )
    test_results.to_excel(test_pred_file, index=False)

    all_pred_file = None
    if SAVE_ALL_PREDICTIONS:
        X_all = prepare_all_features(df, prop, desc_variant, prep)
        y_pred_all = model.predict(X_all)

        keep_cols = [c for c in ["Preferred_name", "NAME", "SMILES", "Training/Test"] if c in df.columns]
        all_results = df[keep_cols].copy()

        target_col = f"{prop}-Measured"
        if target_col in df.columns:
            all_results[target_col] = df[target_col]

        all_results[f"{prop}_{desc_variant}_Prediction"] = y_pred_all
        all_results["split_mode"] = SPLIT_MODE

        all_pred_file = os.path.join(
            OUTPUT_DIR,
            f"{safe_prop}_{desc_variant}_{SPLIT_MODE}_all_predictions.xlsx",
        )
        all_results.to_excel(all_pred_file, index=False)

    return {
        "property": prop,
        "descriptor": desc_variant,
        "split_mode": SPLIT_MODE,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "descriptor_dim": int(X_train.shape[1]),
        "test_pred_path": test_pred_file,
        "all_pred_path": all_pred_file,
    }

In [8]:
# ============================================================
# Run
# ============================================================

metrics_rows = []

for prop in PROPERTY_SHEETS:
    # Make sure PWAV runs before PWAV_SHAP64
    ordered_variants = []
    for d in DESC_VARIANTS:
        if d == "PWAV_SHAP64":
            continue
        ordered_variants.append(d)

    if "PWAV_SHAP64" in DESC_VARIANTS:
        ordered_variants.append("PWAV_SHAP64")

    for desc_variant in ordered_variants:
        result = train_xgb_and_score(prop, desc_variant)
        if result is not None:
            metrics_rows.append(result)

metrics_df = pd.DataFrame(metrics_rows)

summary_file = os.path.join(
    OUTPUT_DIR,
    f"xgboost_summary_{SPLIT_MODE}_PWAV_SHAP64.csv",
)
metrics_df.to_csv(summary_file, index=False)

print("\nFinished all XGBoost runs.")
print(metrics_df)
print(f"\nSaved summary to: {summary_file}")


Processing: Log VP | MACCS | scaffold
Input: Descriptors Data/LogVP_MACCS.parquet
MAE : 1.4332
RMSE: 2.0142
R2  : 0.5850
Descriptor dim: 167

Processing: Log VP | Morgan | scaffold
Input: Descriptors Data/LogVP_Morgan.parquet
MAE : 2.1233
RMSE: 2.6840
R2  : 0.2630
Descriptor dim: 1024

Processing: Log VP | PWAV | scaffold
Input: Descriptors Data/LogVP_PWAV_raw.parquet
MAE : 1.1703
RMSE: 1.5995
R2  : 0.7382
Descriptor dim: 120

Processing: Log VP | PWAV_SHAP64 | scaffold
Input: Descriptors Data/LogVP_PWAV_raw.parquet
Generating SHAP64 indices from full PWAV feature vector...
Saved SHAP64 indices: XGBoost Results Revision/LogVP_PWAV_SHAP64_scaffold_top64_indices.json
MAE : 1.1189
RMSE: 1.5328
R2  : 0.7597
Descriptor dim: 64

Processing: MP | MACCS | scaffold
Input: Descriptors Data/MP_MACCS.parquet
MAE : 40.5727
RMSE: 52.5531
R2  : 0.4996
Descriptor dim: 167

Processing: MP | Morgan | scaffold
Input: Descriptors Data/MP_Morgan.parquet
MAE : 47.2850
RMSE: 61.2950
R2  : 0.3192
Descriptor 